In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

# -----------------------------
# 1. Load dataset
# -----------------------------
df = pd.read_csv("train.csv")

texts = df['comment_text'].astype(str).tolist()

# Multi-label targets (6 columns)
labels = df[['toxic','severe_toxic','obscene','threat','insult','identity_hate']].values

# -----------------------------
# 2. Tokenization
# -----------------------------
MAX_WORDS = 20000       # vocab size
MAX_SEQUENCE_LEN = 150  # sequence length

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=MAX_SEQUENCE_LEN, padding='post', truncating='post')

y = labels

# -----------------------------
# 3. Train-Test Split
# -----------------------------
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# -----------------------------
# 4. Build the Model
# -----------------------------
model = Sequential([
    Embedding(input_dim=MAX_WORDS+1, 
              output_dim=128, 
              input_length=MAX_SEQUENCE_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(6, activation='sigmoid')   # 6 outputs for multi-label classification
])

model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

# -----------------------------
# 5. Train the Model
# -----------------------------
history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=128,
    validation_data=(X_val, y_val)
)

# -----------------------------
# 6. Evaluate
# -----------------------------
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.4f}")


/Users/midhunmanu/Developer/machine_learning/tensorflow-project/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 205s 205ms/step - accuracy: 0.6481 - loss: 0.1810 - val_accuracy: 0.9934 - val_loss: 0.1400
Epoch 2/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 194s 194ms/step - accuracy: 0.9737 - loss: 0.1257 - val_accuracy: 0.9941 - val_loss: 0.0581
Epoch 3/3
998/998 ━━━━━━━━━━━━━━━━━━━━ 197s 197ms/step - accuracy: 0.9928 - loss: 0.0567 - val_accuracy: 0.9941 - val_loss: 0.0619
998/998 ━━━━━━━━━━━━━━━━━━━━ 29s 29ms/step - accuracy: 0.9943 - loss: 0.0615
Validation Accuracy: 0.9941


In [5]:
# -----------------------------
# 6. Make Predictions
# -----------------------------
def predict_comment(text, tokenizer, model, max_len=150):
    # Convert text -> sequence
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    
    # Predict
    pred = model.predict(padded)[0]
    
    # Map results to labels
    labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
    results = {label: float(round(p, 3)) for label, p in zip(labels, pred)}
    return results

# Example predictions
examples = [
    "I hate you, you are the worst!",
    "Have a great day my friend 😊",
    "I will find you and hurt you",
    "I Love You",
    "I will kill you"
]

for text in examples:
    print(f"\nComment: {text}")
    print(predict_comment(text, tokenizer, model))



Comment: I hate you, you are the worst!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
{'toxic': 0.9810000061988831, 'severe_toxic': 0.21400000154972076, 'obscene': 0.878000020980835, 'threat': 0.03099999949336052, 'insult': 0.7379999756813049, 'identity_hate': 0.15800000727176666}

Comment: Have a great day my friend 😊
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
{'toxic': 0.008999999612569809, 'severe_toxic': 0.0, 'obscene': 0.0010000000474974513, 'threat': 0.0, 'insult': 0.0010000000474974513, 'identity_hate': 0.0}

Comment: I will find you and hurt you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
{'toxic': 0.164000004529953, 'severe_toxic': 0.0010000000474974513, 'obscene': 0.029999999329447746, 'threat': 0.006000000052154064, 'insult': 0.05000000074505806, 'identity_hate': 0.013000000268220901}

Comment: I Love You
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
{'toxic': 0.05999999865889549, 'severe_toxic': 0.0, 'obscene': 0.007000000216066837, 'threat': 0.0010000000474974513, 'insult': 0.012000000104308128, 'i

model.save("toxic_comment.h5")